# Project Ascalon — Kaggle 2x T4 Unsloth QLoRA Training Notebook
### Models: Ascalon (Qwen3.5-4B) & CLM / AER-Core (Qwen2.5-1.5B)
This notebook fine-tunes specialist cognitive agents on ShortForge / FactoryOS verified situation packets using Unsloth 4-bit QLoRA on dual NVIDIA Tesla T4 GPUs.

In [ ]:
# Step 1: Install Unsloth and training dependencies
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 4096
dtype = None  # None for auto detection. Float16 for Tesla T4
load_in_4bit = True

# Set base model: Qwen/Qwen2.5-1.5B-Instruct (for CLM) or Qwen/Qwen2.5-3B-Instruct / 4B
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
# Step 2: Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# Step 3: Format SFT Dataset
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Load clean SFT dataset (clm_qwen25_15b_sft.jsonl or ascalon_qwen35_4b_sft.jsonl)
dataset = load_dataset("json", data_files="training/ascalon/data/sft/clm_qwen25_15b_sft.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# Step 4: SFTTrainer Configuration
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs/ascalon_checkpoints",
    ),
)

# Execute training
trainer_stats = trainer.train()

In [ ]:
# Step 5: Save LoRA & Export to GGUF
model.save_pretrained("training/ascalon/outputs/ascalon_lora_adapter")
tokenizer.save_pretrained("training/ascalon/outputs/ascalon_lora_adapter")

# Export directly to GGUF format for Ollama (Q4_K_M)
model.save_pretrained_gguf("training/ascalon/outputs/ascalon_gguf", tokenizer, quantization_method="q4_k_m")
print("Training & GGUF export complete!")